# Epithelial annotation

Takes the epithelial compartment out of `compartments.ipynb` and assigns
`cell_type` labels.

Two clustering passes, in this order:

1. **Cluster, inspect, remove junk.** Low-quality and misassigned clusters are
   identified by eye and dropped.
2. **Re-cluster the cells that remain, then annotate.** Removing cells changes
   the neighbourhood graph, so the graph and UMAP are rebuilt before the
   clustering that the labels are assigned to.

Set `DATA` in `config.py` before running.

In [ ]:
import pandas as pd
import scanpy as sc

from config import DATA

SEED = 0

## 1. Load the epithelial compartment

Written by `compartments.ipynb`.

In [ ]:
adata = sc.read_h5ad(DATA / 'epithelial.h5ad')
print(adata.shape)

## 2. First pass - cluster to find the junk

`n_neighbors=20`, lower than the 30 used on the full object. This compartment
is an order of magnitude smaller, and a neighbourhood that large would smooth
over real structure.

> **The resolution below is not a principled value.** It is tuned by eye, and
> the only thing it has to achieve is putting the cells you want to remove into
> a cluster of their own - not split across several, and not mixed in with a
> real population. That might happen at 0.6, it might take 1.2. Raise it until
> the junk separates cleanly, and no further.

In [ ]:
sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=20, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)

# tune this by eye until the junk sits in its own cluster - see above
sc.tl.leiden(adata, resolution=0.6, random_state=SEED)

print(f"{adata.obs['leiden'].nunique()} clusters")
sc.pl.umap(adata, color='leiden', legend_loc='on data', frameon=False)

Check the clusters before deciding what to drop. A cluster is junk when it has
no distinguishing marker of its own, or when it co-expresses markers of another
compartment - residual `Pecam1` endothelium is the usual offender after the
split, and this is where it gets caught.

In [ ]:
sc.pl.umap(
    adata,
    color=['Pecam1', 'Ptprc', 'Pdgfra', 'n_genes_by_counts'],
    frameon=False
)

## 3. Drop the junk clusters

> **Cluster numbers here belong to the first pass only**, at whatever
> resolution was used above. Re-running this notebook renumbers them. Read the
> plots and rebuild this list.

In [ ]:
JUNK = ['16']   # no distinguishing marker

n_before = adata.n_obs
adata = adata[~adata.obs['leiden'].isin(JUNK)].copy()

print(f'{n_before:,} -> {adata.n_obs:,} cells ({n_before - adata.n_obs:,} dropped)')

## 4. Second pass - re-cluster what is left

The graph and UMAP are rebuilt from scratch. Dropping cells changes every
cell's neighbourhood, so re-using the old graph would annotate against
structure that no longer exists.

The resolution here is chosen the same way as the first - by eye, raised until
the populations below separate and stopped before they fragment. 0.6 is what
that turned out to be for this compartment; it is not a default worth copying.

In [ ]:
sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=20, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)

sc.tl.leiden(adata, resolution=0.6, random_state=SEED)

print(f"{adata.obs['leiden'].nunique()} clusters")
sc.pl.umap(adata, color='leiden', legend_loc='on data', frameon=False)

## 5. Markers

The evidence for the mapping below:

| Marker | Identifies |
| --- | --- |
| `Sftpc` alone | AT2 |
| `Sftpc` + `Lcn2` | Activated AT2 |
| `Ager`, `Hopx` | AT1 |
| `Cldn4` | Intermediate Cell |
| `Scgb1a1` | Secretory Cell |
| `Muc5b` | Goblet Cell |
| `Foxj1` | Ciliated Cell |
| `Trp63` | Basal Cell |
| `Mki67` | cycling |

**Transitional AT2-AT1** is not called from any single marker. It is read from
the combined picture - cells sitting between the AT2 and AT1 states, carrying
neither identity cleanly. This is a judgement made across the whole set of
plots, not a threshold on one gene.

In [ ]:
MARKERS = ['Sftpc', 'Lcn2', 'Ager', 'Hopx', 'Cldn4',
           'Scgb1a1', 'Muc5b', 'Foxj1', 'Trp63', 'Mki67']

sc.pl.umap(adata, color=MARKERS, frameon=False)

Ranked marker genes per cluster, as a cross-check on the UMAPs above. Clusters
whose top genes do not match the call below are worth revisiting.

The table is the useful form - one column per cluster, ranked down the rows, so
an unfamiliar cluster can be identified by reading its column.

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method='t-test')

# one column per cluster, top 100 genes down the rows - this is what the calls
# below were read from
top_genes = pd.DataFrame(adata.uns['rank_genes_groups']['names']).head(100)
top_genes.head(30)

## 6. Assign cell types

Read off the marker plots and the ranked genes above. Several clusters carry
the same label - Leiden splits the large populations further than the biology
justifies, and merging them here is deliberate.

> These numbers belong to the second clustering pass in this notebook. They are
> a record of the run that produced the deposited object, not something to
> trust after a re-run.

In [ ]:
CELL_TYPES = {
    '0':  'Activated AT2',
    '1':  'AT2',
    '2':  'Transitional AT2-AT1 Cell',
    '3':  'AT1',
    '4':  'Ciliated Cell',
    '5':  'Secretory Cell',
    '6':  'Intermediate Cell',
    '7':  'Intermediate Cell',
    '8':  'AT1',
    '9':  'Cyling AT2',
    '10': 'Secretory Cell',
    '11': 'Basal Cell',
}

adata.obs['cell_type'] = adata.obs['leiden'].map(CELL_TYPES).astype('category')
print(adata.obs['cell_type'].value_counts().to_string())

In [ ]:
sc.pl.umap(adata, color='cell_type', frameon=False)

Composition per condition - the reason the annotation exists. Any label that
appears in only one sample is worth a second look before it reaches a figure.

In [ ]:
sc.pl.umap(adata, color=['type', 'age'], frameon=False)

## 7. Save

In [ ]:
adata.write_h5ad(DATA / 'epithelial_annotated.h5ad')
print(f'{adata.n_obs:,} cells, {adata.obs["cell_type"].nunique()} cell types')